# Width Detection Experiments

This notebook explores different heuristics for detecting the optimal 2D width when reshaping FPGA bitstreams.

## Goals

1. Load a sample `.sof` file
2. Convert to bit vector
3. Test multiple width candidates
4. Visualize autocorrelation and entropy metrics
5. Compare with known good widths (if available)

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from preprocess.sof_to_bits import bytes_to_bits
from preprocess.extract_sof_sections import extract_design_section
from preprocess.guess_width_autocorr import score_width, bits_to_image

## Load sample bitstream

In [ ]:
# TODO: Replace with actual .sof path
sof_path = Path("../dataset/raw_sof/example.sof")

if sof_path.exists():
    raw = sof_path.read_bytes()
    design = extract_design_section(raw)
    bits = bytes_to_bits(design)
    print(f"Loaded {len(bits)} bits from {sof_path}")
else:
    print(f"File not found: {sof_path}")
    print("Using synthetic data for demonstration")
    bits = np.random.randint(0, 2, size=1000000, dtype=np.uint8)

## Test candidate widths

In [ ]:
candidates = [256, 384, 512, 768, 1024, 1536, 2048, 3072, 4096]
scores = []

for w in candidates:
    s = score_width(bits, w)
    scores.append(s)
    print(f"Width {w:4d}: score = {s:.6f}")

best_idx = np.argmax(scores)
best_width = candidates[best_idx]
print(f"\nBest width: {best_width}")

## Visualize results

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(candidates, scores, 'o-')
plt.axvline(best_width, color='r', linestyle='--', label=f'Best: {best_width}')
plt.xlabel('Width (bits)')
plt.ylabel('Structure Score')
plt.title('Width Detection Scores')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
img = bits_to_image(bits, best_width)
plt.imshow(img[:500, :], cmap='gray', aspect='auto')
plt.title(f'Bitstream (width={best_width}, first 500 rows)')
plt.xlabel('Bit position')
plt.ylabel('Frame')
plt.colorbar(label='Bit value')

plt.tight_layout()
plt.show()

## Next Steps

- Try more sophisticated scoring metrics (FFT autocorrelation, mutual information)
- Compare with known frame widths from Intel documentation
- Test on multiple device families (Arria 10, Stratix 10, Cyclone V)